In [1]:
from utils_shiprocket import transcribe_item, prepare_data

train_df, val_df, _ = prepare_data(train_examples=None)

Resolving data files:   0%|          | 0/83 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/83 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/82 [00:00<?, ?it/s]

In [2]:
from utils_shiprocket import AudioDataset,gru_collate_fn,AudioGRU
from torch.utils.data import Dataset, DataLoader

features =  ("log-mel-spectrogram",) # features are low... deliberately

train_dataset = AudioDataset(train_df,features)
val_dataset = AudioDataset(val_df,features)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=gru_collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=gru_collate_fn,
)


AudioGRU(
  (cnn): Sequential(
    (0): Conv1d(128, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): ReLU()
    (2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (3): ReLU()
  )
  (gru): GRU(64, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)

In [17]:
def predict_batch(loader):
    results = []
    with torch.no_grad():
        for X, lengths, y in loader:
            logits = model(X, lengths)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).long()

            for p, pr, true in zip(preds.cpu().tolist(), probs.cpu().tolist(), y.cpu().tolist()):
                results.append({"prediction": p, "probability": pr, "label": true})
                print({"prediction": p, "probability": pr, "label": true})
    return results

predict_batch(val_loader)

{'prediction': 0, 'probability': 0.0015863305889070034, 'label': 0.0}
{'prediction': 1, 'probability': 0.8964577913284302, 'label': 1.0}
{'prediction': 0, 'probability': 0.08486156165599823, 'label': 0.0}
{'prediction': 0, 'probability': 0.02669471502304077, 'label': 0.0}
{'prediction': 1, 'probability': 0.7430356740951538, 'label': 1.0}
{'prediction': 1, 'probability': 0.6446701884269714, 'label': 0.0}
{'prediction': 0, 'probability': 0.048280585557222366, 'label': 0.0}
{'prediction': 1, 'probability': 0.9137566089630127, 'label': 0.0}
{'prediction': 1, 'probability': 0.7768241167068481, 'label': 1.0}
{'prediction': 1, 'probability': 0.895507276058197, 'label': 1.0}
{'prediction': 0, 'probability': 0.04538841173052788, 'label': 0.0}
{'prediction': 0, 'probability': 0.0975426584482193, 'label': 1.0}
{'prediction': 0, 'probability': 0.09365642070770264, 'label': 1.0}
{'prediction': 0, 'probability': 0.19926650822162628, 'label': 0.0}
{'prediction': 1, 'probability': 0.6986546516418457, 

KeyboardInterrupt: 

In [13]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from utils_shiprocket import evaluate_binary_classifier
from tqdm.auto import tqdm
import torch

norm_stats = torch.load("norm_stats.pt", map_location="cpu")
train_mean = norm_stats["train_mean"]
train_std = norm_stats["train_std"]

model = AudioGRU(128, train_mean, train_std, hidden_size=64, conv_channels=64)
checkpoint = torch.load("interrupted_checkpoint.pt", map_location="cpu")
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

ALPHA = 0.5
BETA = 0.5
THRESHOLD = 0.5

MODEL_DIR = "/Users/akshat.khatri/PycharmProjects/Shiprocket_final/kaggle/working/muril-endpoint-clf/checkpoint-3500"
TRANSCRIPTS_PATH = "/Users/akshat.khatri/PycharmProjects/Shiprocket_final/transcripts/merged_output_val.jsonl"
device = "cuda" if torch.cuda.is_available() else "cpu"

audio_model = model
audio_model.eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
text_model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device).eval()


def load_transcript_lookup(path):
    lookup = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            lookup[row["id"]] = row["text"]
    return lookup


def build_texts_for_val_df(val_df, transcript_lookup):
    texts, missing_ids = [], []
    for _id in val_df["id"]:
        if _id in transcript_lookup:
            texts.append(transcript_lookup[_id])
        else:
            texts.append(None)
            missing_ids.append(_id)
    return texts, missing_ids


def get_audio_probs(loader):
    results = []
    with torch.no_grad():
        for X, lengths, y in tqdm(loader, desc="audio"):
            logits = audio_model(X, lengths)
            probs = torch.sigmoid(logits)
            for pr, true in zip(probs.cpu().tolist(), y.cpu().tolist()):
                results.append({"prob": pr, "label": true})
    return results


def get_text_probs(texts, batch_size=32):
    probs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="text"):
            batch = texts[i:i + batch_size]
            enc = tokenizer(batch, truncation=True, padding=True, max_length=256, return_tensors="pt").to(device)
            logits = text_model(**enc).logits
            p = torch.softmax(logits, dim=-1)
            probs.extend(p[:, 1].cpu().tolist())
    return probs


def ensemble_predict(audio_loader, texts, missing_ids):
    audio_results = get_audio_probs(audio_loader)
    assert len(audio_results) == len(texts), (
        f"Length mismatch: audio={len(audio_results)} vs texts={len(texts)}. "
        "val_loader must be built with shuffle=False and iterate in val_df's row order."
    )
    if missing_ids:
        print(f"WARNING: {len(missing_ids)} id(s) from val_df were not found in transcripts file:")
        for mid in missing_ids:
            print(f"  missing id: {mid}")

    valid_indices = [i for i, t in enumerate(texts) if t is not None]
    valid_probs = get_text_probs([texts[i] for i in valid_indices])
    text_probs_by_index = dict(zip(valid_indices, valid_probs))

    y_true, y_pred = [], []
    for i, a in enumerate(audio_results):
        if i not in text_probs_by_index:
            continue
        combined_prob = ALPHA * a["prob"] + BETA * text_probs_by_index[i]
        y_true.append(int(a["label"]))
        y_pred.append(int(combined_prob >= THRESHOLD))
    return y_true, y_pred


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [15]:
transcript_lookup = load_transcript_lookup(TRANSCRIPTS_PATH)
texts, missing_ids = build_texts_for_val_df(val_df, transcript_lookup)
y_true, y_pred = ensemble_predict(val_loader, texts, missing_ids)
metrics = evaluate_binary_classifier(y_true, y_pred)
print(f"accuracy  : {metrics['accuracy']:.4f}")
print(f"precision : {metrics['precision']:.4f}")
print(f"recall    : {metrics['recall']:.4f}")
print(f"f1_score  : {metrics['f1_score']:.4f}")

audio:   0%|          | 0/487 [00:00<?, ?it/s]

text:   0%|          | 0/244 [00:00<?, ?it/s]

accuracy  : 0.9576
precision : 0.9444
recall    : 0.9702
f1_score  : 0.9571


In [10]:
print(val_df["id"][:5])

['c392b019-8df6-4a50-95d1-558f572f3fe0', 'b832db69-5852-42cf-a7b8-dc945e4c551d', '68691ccc-7624-4862-a71d-bae7d8cbb8ee', '2ba30d30-099a-42cb-a5f3-d78078ef8fd5', '4a7d094a-b215-4dfb-ab4d-202f0ce01356']
